In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv('RecipeNLG_dataset.csv')

In [ ]:
df.head()

In [ ]:
print(df.columns)

In [ ]:
# Drop unnecessary columns
df = df[['title', 'ingredients', 'directions']].dropna()

# Convert ingredients to lowercase
df['ingredients_clean'] = df['ingredients'].str.lower().str.replace(r"[\[\]'\"{}]", '', regex=True)

# Show example of cleaned data
df[['title', 'ingredients_clean', 'directions']].head(3)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Vectorize cleaned ingredient strings
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['ingredients_clean'])

In [ ]:
# Vegetarian Filter

# Define a list of common non-vegetarian ingredients
non_veg_keywords = ['chicken', 'beef', 'pork', 'fish', 'shrimp', 'bacon', 'turkey', 'lamb', 'ham', 'sausage']

# Create vegetarian flag
def is_vegetarian(ingredients):
    return not any(meat in ingredients for meat in non_veg_keywords)

df['vegetarian'] = df['ingredients_clean'].apply(is_vegetarian)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_recipes(user_ingredients, top_n=5, vegetarian_only=False):
    # Preprocess input
    user_ingredients = user_ingredients.lower()
    user_vec = vectorizer.transform([user_ingredients])
    similarity_scores = cosine_similarity(user_vec, tfidf_matrix)
    top_indices = similarity_scores[0].argsort()[::-1]
    
    # Get top matching recipes
    results = df.iloc[top_indices]

    if vegetarian_only:
        results = results[results['vegetarian']]

    return results[['title', 'ingredients_clean', 'directions']].head(top_n)

In [ ]:
# Sample input
input_ingredients = "garlic, salmon, rice"

# Get recommendations
recommendations = recommend_recipes(input_ingredients)

In [ ]:
show_recommendations(recommendations)

In [ ]:
df.to_csv("cleaned.csv", index=False)